# Module 1.1 — Premier graphique avec Claude Code

**Formation Data Mastery** | De la visualisation au Deep Learning

---

Ce notebook est le plus concret du module. On passe de fichiers CSV bruts à des graphiques interactifs en Python, en suivant exactement le processus qu'on suivrait avec Claude Code dans le terminal.

## Objectif

À la fin de ce notebook, on sait :

- Charger un CSV avec pandas
- Explorer rapidement un jeu de données (dimensions, colonnes, types, statistiques)
- Créer 5 types de graphiques avec Plotly : barres, lignes, sunburst, scatter, série temporelle
- Personnaliser l'apparence des graphiques (titres, axes, couleurs)
- Lire et comprendre le code généré

**Données utilisées** :
- `../data/consommation_energie.csv` — mesures d'énergie par bâtiment, zone et type
- `../data/production_industrielle.csv` — données de production par ligne et équipe
- `../data/capteurs_temperature.csv` — relevés IoT de capteurs de température

**Structure des exemples** : pour chaque graphique, on voit le prompt donné à Claude Code, le code généré avec commentaires, le résultat, et ce qu'on apprend.

In [ ]:
# Cellule d'initialisation — importer les bibliothèques utilisées dans tout le notebook
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Chemin vers les données (relatif à ce notebook)
DATA = "../data/"

print("Bibliothèques chargées.")
print(f"pandas  : {pd.__version__}")
import plotly
print(f"plotly  : {plotly.__version__}")

---

## Exemple 1 — Consommation énergétique par bâtiment (graphique en barres)

### Le prompt donné à Claude Code

```
Charge le fichier ../data/consommation_energie.csv.
Explore sa structure : dimensions, colonnes, types, premières lignes.
Calcule la consommation totale en kWh par bâtiment.
Crée un graphique en barres avec plotly express.
Ajoute un titre clair et des labels d'axes en français.
```

### Étape 1.1 — Chargement et exploration du fichier

In [ ]:
# Chargement du CSV dans un DataFrame pandas
# pd.read_csv() lit un fichier CSV et le convertit en tableau structuré (DataFrame)
df_energie = pd.read_csv(DATA + "consommation_energie.csv")

# Dimensions du tableau : (nombre de lignes, nombre de colonnes)
print("Dimensions :", df_energie.shape)
print(f"  → {df_energie.shape[0]} mesures, {df_energie.shape[1]} colonnes")

In [ ]:
# Les 5 premières lignes — pour voir à quoi ressemblent les données
# head() est la première commande à lancer sur tout nouveau jeu de données
df_energie.head()

In [ ]:
# Types de données par colonne
# object = texte, float64 = nombre décimal, int64 = nombre entier
print("Types des colonnes :")
print(df_energie.dtypes)
print()
print("Valeurs uniques par colonne catégorielle :")
print("  batiment    :", df_energie["batiment"].unique())
print("  type_energie:", df_energie["type_energie"].unique())
print("  zone        :", df_energie["zone"].unique())

In [ ]:
# Statistiques descriptives de la colonne numérique kwh
# describe() calcule automatiquement count, moyenne, écart-type, min, quartiles, max
print("Statistiques de la consommation (kWh) :")
print(df_energie["kwh"].describe().round(2))

### Étape 1.2 — Agrégation et graphique en barres

In [ ]:
# Calcul de la consommation totale par bâtiment
# groupby() regroupe les lignes par valeur unique d'une colonne
# ["kwh"].sum() additionne les kWh de chaque groupe
# reset_index() remet l'index à plat pour que le résultat soit un DataFrame normal
conso_par_batiment = (
    df_energie
    .groupby("batiment")["kwh"]
    .sum()
    .reset_index()
    .rename(columns={"kwh": "kwh_total"})
    .sort_values("kwh_total", ascending=False)  # Du plus consommateur au moins
)

print("Consommation totale par bâtiment :")
print(conso_par_batiment.to_string(index=False))

In [ ]:
# Création du graphique en barres avec Plotly Express
# px.bar() génère un graphique interactif directement explorable dans le navigateur
fig = px.bar(
    conso_par_batiment,              # Le DataFrame à visualiser
    x="batiment",                    # Colonne pour l'axe horizontal
    y="kwh_total",                   # Colonne pour l'axe vertical
    title="Consommation énergétique totale par bâtiment (2023-2024)",
    labels={
        "batiment": "Bâtiment",
        "kwh_total": "Consommation totale (kWh)"
    },
    color="batiment",                # Une couleur différente par bâtiment
    color_discrete_sequence=px.colors.qualitative.Set2,  # Palette de couleurs
    text="kwh_total"                 # Affiche la valeur sur chaque barre
)

# Personnalisation de l'affichage des valeurs sur les barres
fig.update_traces(texttemplate="%{text:,.0f} kWh", textposition="outside")

# Ajustement du layout
fig.update_layout(
    showlegend=False,                # La légende est redondante avec les labels de l'axe X
    yaxis_tickformat=",.0f",         # Format des nombres sur l'axe Y (séparateur de milliers)
    plot_bgcolor="white",
    height=500
)

fig.show()

> **Ce qu'on apprend avec cet exemple**
> 
> - `pd.read_csv()` charge n'importe quel CSV en une ligne
> - `groupby().sum()` est le patron fondamental pour agréger des données par catégorie
> - `px.bar()` crée un graphique interactif en quelques paramètres
> - Le graphique est zoomable, les valeurs s'affichent au survol — tout cela sans configuration supplémentaire
> 
> **Lecture du résultat** : Usine_A et Usine_B sont les deux plus gros postes de consommation. Bureau_Central consomme significativement moins, ce qui est cohérent avec sa surface et son activité.

---

## Exemple 2 — Évolution mensuelle de la consommation (graphique en lignes)

### Le prompt donné à Claude Code

```
Sur le même fichier consommation_energie.csv,
parse la colonne date comme datetime,
regroupe par mois et par bâtiment,
crée un graphique en lignes avec une ligne par bâtiment.
Ajoute une ligne de tendance globale.
```

In [ ]:
# Conversion de la colonne date en type datetime
# Sans cette étape, la date est traitée comme du texte et ne peut pas être triée ni rééchantillonnée
df_energie["date"] = pd.to_datetime(df_energie["date"])

# Création d'une colonne mois au format YYYY-MM pour le regroupement
# .dt.to_period("M") extrait la période mensuelle d'une date
df_energie["mois"] = df_energie["date"].dt.to_period("M")

print("Exemple de la colonne mois créée :")
print(df_energie[["date", "mois"]].head())

In [ ]:
# Agrégation par mois ET par bâtiment
# On regroupe sur deux colonnes simultanément pour obtenir
# la consommation de chaque bâtiment pour chaque mois
conso_mensuelle = (
    df_energie
    .groupby(["mois", "batiment"])["kwh"]
    .sum()
    .reset_index()
)

# Conversion de Period en str pour plotly (plotly ne comprend pas le type Period)
conso_mensuelle["mois"] = conso_mensuelle["mois"].astype(str)

print(f"Tableau résultant : {conso_mensuelle.shape[0]} lignes (mois × bâtiments)")
conso_mensuelle.head(8)

In [ ]:
# Graphique en lignes avec une ligne par bâtiment
# color="batiment" crée automatiquement une ligne par valeur unique de la colonne batiment
fig = px.line(
    conso_mensuelle,
    x="mois",
    y="kwh",
    color="batiment",
    title="Évolution mensuelle de la consommation énergétique par bâtiment",
    labels={
        "mois": "Mois",
        "kwh": "Consommation (kWh)",
        "batiment": "Bâtiment"
    },
    markers=True                     # Ajoute des points sur chaque mesure mensuelle
)

# Rotation des labels de l'axe X pour éviter les chevauchements
fig.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor="white",
    yaxis_tickformat=",.0f",
    height=500,
    legend_title_text="Bâtiment"
)

# Ajout d'une grille légère pour faciliter la lecture
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#f0f0f0")
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#f0f0f0")

fig.show()

In [ ]:
# Ajout d'une ligne de tendance — consommation globale tous bâtiments confondus
conso_globale = (
    conso_mensuelle
    .groupby("mois")["kwh"]
    .sum()
    .reset_index()
)

# Graphique avec la tendance globale en pointillés
fig2 = px.line(
    conso_mensuelle,
    x="mois",
    y="kwh",
    color="batiment",
    title="Évolution mensuelle — avec tendance globale",
    labels={"mois": "Mois", "kwh": "Consommation (kWh)", "batiment": "Bâtiment"},
    markers=True
)

# Ajout de la tendance globale comme trace supplémentaire
# go.Scatter() permet d'ajouter des éléments graphiques manuellement
fig2.add_trace(go.Scatter(
    x=conso_globale["mois"],
    y=conso_globale["kwh"],
    mode="lines",
    name="Total (tous bâtiments)",
    line=dict(color="black", width=3, dash="dash")  # Ligne noire en pointillés
))

fig2.update_layout(
    xaxis_tickangle=-45,
    plot_bgcolor="white",
    yaxis_tickformat=",.0f",
    height=500
)

fig2.show()

> **Ce qu'on apprend avec cet exemple**
>
> - `pd.to_datetime()` est indispensable pour travailler avec des dates — c'est un réflexe à adopter
> - `groupby()` sur plusieurs colonnes permet des analyses croisées
> - `px.line()` avec `color=` crée automatiquement autant de traces que de catégories
> - `add_trace()` permet de superposer des éléments sur un graphique existant
>
> **Lecture du résultat** : les pics de consommation en hiver (chauffage) et en été (climatisation) sont visibles. La tendance globale permet de détecter une dérive annuelle éventuelle.

---

## Exemple 3 — Répartition par type d'énergie (graphique sunburst)

### Le prompt donné à Claude Code

```
Toujours sur consommation_energie.csv,
crée un graphique sunburst avec 3 niveaux :
bâtiment > zone > type_energie.
La taille de chaque segment doit refléter la consommation en kWh.
```

### Qu'est-ce qu'un graphique sunburst ?

Le sunburst est un graphique circulaire à plusieurs niveaux — chaque anneau représente un niveau de hiérarchie. Le centre montre le niveau le plus haut (ici : le total), le premier anneau les bâtiments, le second les zones, et l'extérieur les types d'énergie.

C'est particulièrement adapté pour visualiser des données hiérarchiques comme la répartition de coûts ou de consommation à plusieurs niveaux.

In [ ]:
# Agrégation pour le sunburst : bâtiment × zone × type_energie
conso_hierarchique = (
    df_energie
    .groupby(["batiment", "zone", "type_energie"])["kwh"]
    .sum()
    .reset_index()
)

print(f"Nombre de combinaisons bâtiment/zone/énergie : {len(conso_hierarchique)}")
conso_hierarchique.head(10)

In [ ]:
# Graphique sunburst à 3 niveaux de hiérarchie
# path définit l'ordre des niveaux, du plus général (batiment) au plus précis (type_energie)
# values définit la taille de chaque segment

# Palette inspirée des couleurs Anthemion (tons professionnels, bleu/vert/ardoise)
PALETTE_ANTHEMION = [
    "#2C5F8A",  # Bleu profond
    "#3A8A6B",  # Vert ardoise
    "#1E3A5F",  # Bleu marine
    "#4A7C8A",  # Bleu gris
    "#5A8A4A",  # Vert forêt
    "#8A6B3A",  # Brun chaud
    "#6B3A8A",  # Violet discret
    "#8A3A4A",  # Bordeaux
]

fig = px.sunburst(
    conso_hierarchique,
    path=["batiment", "zone", "type_energie"],   # Hiérarchie : 3 niveaux
    values="kwh",
    title="Répartition de la consommation énergétique<br>Bâtiment > Zone > Type d'énergie",
    color="batiment",
    color_discrete_sequence=PALETTE_ANTHEMION
)

fig.update_traces(
    textinfo="label+percent parent",             # Affiche le label et le % du niveau parent
    hovertemplate="<b>%{label}</b><br>Consommation : %{value:,.0f} kWh<br>Part : %{percentRoot:.1%}<extra></extra>"
)

fig.update_layout(height=600)
fig.show()

In [ ]:
# Lecture chiffrée du sunburst — tableau récapitulatif
recap = (
    df_energie
    .groupby(["batiment", "type_energie"])["kwh"]
    .sum()
    .reset_index()
    .pivot(index="batiment", columns="type_energie", values="kwh")
    .round(0)
)
recap["total"] = recap.sum(axis=1)
recap["part_electricite_%"] = (recap["electricite"] / recap["total"] * 100).round(1)

print("Répartition électricité / gaz par bâtiment :")
print(recap.to_string())

> **Ce qu'on apprend avec cet exemple**
>
> - Le sunburst est un graphique de choix pour les données hiérarchiques (coûts, consommation, organisation)
> - `path=` dans `px.sunburst()` définit les niveaux en une seule liste — très concis
> - La combinaison `groupby` + `pivot` permet de reformater un tableau pour une lecture directe
> - Les graphiques Plotly sont interactifs : un clic sur un bâtiment zoome sur ce niveau
>
> **Lecture du résultat** : Usine_A et Usine_B montrent la répartition gaz/électricité par zone, ce qui permet d'identifier les postes les plus accessibles pour des actions d'efficacité énergétique.

---

## Exemple 4 — Production vs défauts (nuage de points)

### Le prompt donné à Claude Code

```
Charge production_industrielle.csv.
Crée un nuage de points (scatter) :
- X : quantite_produite
- Y : quantite_defauts
- Couleur : operateur
- Taille des points : temps_arret_min
Ajoute une ligne de tendance.
```

In [ ]:
# Chargement du fichier de production
df_prod = pd.read_csv(DATA + "production_industrielle.csv")

print("Dimensions :", df_prod.shape)
print("\nColonnes et types :")
print(df_prod.dtypes)
print("\nPremières lignes :")
df_prod.head()

In [ ]:
# Statistiques de base sur la production et les défauts
print("Statistiques production / défauts :")
print(df_prod[["quantite_produite", "quantite_defauts", "temps_arret_min"]].describe().round(2))
print()

# Calcul du taux de défauts
df_prod["taux_defauts_pct"] = (df_prod["quantite_defauts"] / df_prod["quantite_produite"] * 100).round(2)
print("Taux de défauts moyen par opérateur :")
print(
    df_prod.groupby("operateur")["taux_defauts_pct"]
    .mean()
    .round(2)
    .to_string()
)

In [ ]:
# Nuage de points : production vs défauts
# Chaque point représente une session de production (une ligne du CSV)
# La couleur distingue les équipes, la taille des points reflète le temps d'arrêt

fig = px.scatter(
    df_prod,
    x="quantite_produite",
    y="quantite_defauts",
    color="operateur",               # Une couleur par équipe
    size="temps_arret_min",          # Taille proportionnelle au temps d'arrêt
    size_max=20,                     # Taille maximale des points
    title="Production vs Défauts par équipe<br>Taille des points = temps d'arrêt (min)",
    labels={
        "quantite_produite": "Quantité produite (unités)",
        "quantite_defauts": "Quantité de défauts",
        "operateur": "Équipe",
        "temps_arret_min": "Temps d'arrêt (min)"
    },
    trendline="ols",                 # OLS = régression linéaire (Ordinary Least Squares)
    trendline_scope="overall",       # Une seule ligne de tendance pour toutes les équipes
    opacity=0.7,                     # Légère transparence pour voir les points superposés
    hover_data=["ligne", "produit", "date"]  # Données supplémentaires au survol
)

fig.update_layout(
    plot_bgcolor="white",
    height=550
)
fig.update_xaxes(showgrid=True, gridcolor="#f0f0f0")
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0")

fig.show()

In [ ]:
# Analyse complémentaire : taux de défauts par ligne de production
# Pour comprendre si certaines lignes sont plus problématiques
taux_par_ligne = (
    df_prod
    .groupby("ligne")
    .agg(
        production_totale=("quantite_produite", "sum"),
        defauts_totaux=("quantite_defauts", "sum"),
        arret_moyen_min=("temps_arret_min", "mean")
    )
    .assign(taux_defauts_pct=lambda df: (df["defauts_totaux"] / df["production_totale"] * 100).round(2))
    .reset_index()
)

print("Performance par ligne de production :")
print(taux_par_ligne.to_string(index=False))

In [ ]:
# Graphique comparatif du taux de défauts par ligne et par équipe
taux_croise = (
    df_prod
    .groupby(["ligne", "operateur"])
    .agg(
        taux_defauts=("taux_defauts_pct", "mean")
    )
    .reset_index()
)

fig2 = px.bar(
    taux_croise,
    x="ligne",
    y="taux_defauts",
    color="operateur",
    barmode="group",                 # Barres côte à côte (vs "stack" pour barres empilées)
    title="Taux de défauts moyen par ligne et par équipe",
    labels={
        "ligne": "Ligne de production",
        "taux_defauts": "Taux de défauts (%)",
        "operateur": "Équipe"
    }
)
fig2.update_layout(plot_bgcolor="white", height=450)
fig2.show()

> **Ce qu'on apprend avec cet exemple**
>
> - `px.scatter()` avec `size=` encode une 4e dimension dans la taille des points
> - `trendline="ols"` ajoute une régression linéaire automatiquement (nécessite `statsmodels`)
> - `hover_data=` enrichit l'infobulle au survol sans alourdir le graphique
> - `.agg()` permet de calculer plusieurs agrégats en un seul appel, plus lisible que des `groupby` successifs
>
> **Lecture du résultat** : la ligne de tendance montre si les défauts augmentent proportionnellement à la production (corrélation positive attendue). Les points anormalement grands (temps d'arrêt élevé) méritent une investigation spécifique.

---

## Exemple 5 — Capteurs IoT en temps réel (série temporelle)

### Le prompt donné à Claude Code

```
Charge capteurs_temperature.csv.
Crée un graphique en lignes de l'évolution de la température
pour chaque capteur (T01 à T05) sur la journée.
Mets en évidence la dérive anormale du capteur T03.
```

In [ ]:
# Chargement des données capteurs
df_capteurs = pd.read_csv(DATA + "capteurs_temperature.csv")

# Conversion du timestamp en datetime — étape essentielle pour les séries temporelles
df_capteurs["timestamp"] = pd.to_datetime(df_capteurs["timestamp"])

print("Dimensions :", df_capteurs.shape)
print("\nPlage temporelle :", df_capteurs["timestamp"].min(), "→", df_capteurs["timestamp"].max())
print("\nCapteurs disponibles :", sorted(df_capteurs["capteur_id"].unique()))
print("\nTemperature stats par capteur :")
print(
    df_capteurs.groupby("capteur_id")["temperature_c"]
    .agg(["mean", "min", "max", "std"])
    .round(2)
    .to_string()
)

In [ ]:
# Graphique de série temporelle : tous les capteurs
# Chaque capteur = une ligne de couleur différente
fig = px.line(
    df_capteurs,
    x="timestamp",
    y="temperature_c",
    color="capteur_id",
    title="Relevés de température — Capteurs T01 à T05 (1 mars 2024)",
    labels={
        "timestamp": "Heure",
        "temperature_c": "Température (°C)",
        "capteur_id": "Capteur"
    }
)

fig.update_layout(
    plot_bgcolor="white",
    height=500,
    xaxis_tickformat="%H:%M",        # Affiche uniquement l'heure sur l'axe X
)
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0")

fig.show()

In [ ]:
# Détection et mise en évidence de l'anomalie T03
# Méthode : identifier les points en dehors de la plage normale (moyenne ± 2 écarts-types)

# Calcul des statistiques de référence sur T01/T02/T04/T05 (capteurs normaux)
capteurs_reference = df_capteurs[df_capteurs["capteur_id"] != "T03"].copy()
temp_moyenne = capteurs_reference["temperature_c"].mean()
temp_ecart = capteurs_reference["temperature_c"].std()

seuil_haut = temp_moyenne + 2 * temp_ecart
seuil_bas  = temp_moyenne - 2 * temp_ecart

print(f"Plage normale (capteurs de référence) :")
print(f"  Moyenne : {temp_moyenne:.2f} °C")
print(f"  Écart-type : {temp_ecart:.2f} °C")
print(f"  Seuil haut (moy + 2σ) : {seuil_haut:.2f} °C")
print(f"  Seuil bas  (moy - 2σ) : {seuil_bas:.2f} °C")

# Identifier les points anormaux de T03
t03 = df_capteurs[df_capteurs["capteur_id"] == "T03"].copy()
t03_anormal = t03[t03["temperature_c"] > seuil_haut]
print(f"\nPoints anormaux sur T03 : {len(t03_anormal)} sur {len(t03)} mesures")
print(f"Température maximale T03 : {t03['temperature_c'].max():.2f} °C")

In [ ]:
# Graphique annoté — mise en évidence de l'anomalie T03

fig = px.line(
    df_capteurs,
    x="timestamp",
    y="temperature_c",
    color="capteur_id",
    title="Relevés de température — Anomalie détectée sur T03",
    labels={
        "timestamp": "Heure",
        "temperature_c": "Température (°C)",
        "capteur_id": "Capteur"
    },
    color_discrete_map={
        "T01": "#4A90D9",   # Bleu
        "T02": "#5BA85C",   # Vert
        "T03": "#E05C5C",   # Rouge — attire l'attention sur la dérive
        "T04": "#7B68EE",   # Violet
        "T05": "#FFA500",   # Orange
    }
)

# Zone de normalité en fond (bande grise entre seuil bas et seuil haut)
fig.add_hrect(
    y0=seuil_bas, y1=seuil_haut,
    fillcolor="lightgreen", opacity=0.15,
    annotation_text="Plage normale (±2σ)",
    annotation_position="top right"
)

# Ligne du seuil haut
fig.add_hline(
    y=seuil_haut,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Seuil alerte : {seuil_haut:.1f} °C",
    annotation_position="top left"
)

# Points d'anomalie sur T03 en surbrillance
fig.add_scatter(
    x=t03_anormal["timestamp"],
    y=t03_anormal["temperature_c"],
    mode="markers",
    marker=dict(color="red", size=8, symbol="x"),
    name="Anomalie T03",
    showlegend=True
)

fig.update_layout(
    plot_bgcolor="white",
    height=550,
    xaxis_tickformat="%H:%M",
    legend_title_text="Capteur"
)
fig.update_yaxes(showgrid=True, gridcolor="#f0f0f0")

fig.show()

> **Ce qu'on apprend avec cet exemple**
>
> - `color_discrete_map=` permet d'assigner des couleurs spécifiques à chaque catégorie
> - `add_hrect()` et `add_hline()` ajoutent des annotations de référence visuelles
> - La détection d'anomalie par seuil statistique (±2σ) est une méthode simple mais efficace
> - `add_scatter()` permet de superposer des marqueurs spécifiques (ici : les points anormaux)
>
> **Lecture du résultat** : T03 montre une dérive croissante à partir de 8h00, atteignant 30.56 °C alors que les autres capteurs restent sous 22 °C. Ce profil typique indique soit un capteur défectueux, soit une source de chaleur locale à investiguer.

---

## Comprendre le code — Décryptage des concepts clés

Cette section explique les briques de base utilisées dans tous les exemples précédents.

### Qu'est-ce qu'un DataFrame ?

Un DataFrame pandas est un tableau de données en mémoire. On peut le visualiser comme une feuille Excel — avec des lignes et des colonnes — mais accessible via du code Python.

Chaque colonne est une **Series** (une liste typée). Chaque ligne a un **index** (numéro par défaut, ou une valeur choisie). La puissance du DataFrame vient des opérations qu'on peut lui appliquer en une ligne de code.

In [ ]:
# Anatomie d'un DataFrame — opérations fondamentales
df = pd.read_csv(DATA + "consommation_energie.csv")

print("--- Sélection d'une colonne (retourne une Series) ---")
print(df["batiment"].head(3))

print("\n--- Sélection de plusieurs colonnes (retourne un DataFrame) ---")
print(df[["batiment", "kwh"]].head(3))

print("\n--- Filtrage de lignes (condition booléenne) ---")
usine_a = df[df["batiment"] == "Usine_A"]
print(f"Lignes Usine_A : {len(usine_a)} sur {len(df)}")

print("\n--- Opération sur une colonne ---")
df["kwh_mwh"] = df["kwh"] / 1000   # Conversion kWh → MWh
print(df[["kwh", "kwh_mwh"]].head(3))

### Qu'est-ce que groupby ?

`groupby` applique une logique « diviser / appliquer / combiner » :
1. **Diviser** : scinde le DataFrame en groupes selon les valeurs d'une colonne
2. **Appliquer** : calcule une opération sur chaque groupe (somme, moyenne, comptage…)
3. **Combiner** : rassemble les résultats dans un nouveau DataFrame

In [ ]:
# Illustration de groupby avec différentes opérations
df = pd.read_csv(DATA + "consommation_energie.csv")

print("--- Somme par bâtiment ---")
print(df.groupby("batiment")["kwh"].sum().round(0).to_string())

print("\n--- Moyenne par bâtiment ---")
print(df.groupby("batiment")["kwh"].mean().round(2).to_string())

print("\n--- Nombre de mesures par bâtiment ---")
print(df.groupby("batiment")["kwh"].count().to_string())

print("\n--- Plusieurs agrégats simultanément ---")
print(
    df.groupby("batiment")["kwh"]
    .agg(["sum", "mean", "count"])
    .rename(columns={"sum": "total_kwh", "mean": "moyenne_kwh", "count": "nb_mesures"})
    .round(1)
    .to_string()
)

### Qu'est-ce que Plotly Express ?

Plotly Express (`px`) est une interface simplifiée pour créer des graphiques Plotly. La logique est toujours la même :

```python
px.type_de_graphique(
    dataframe,       # Le tableau de données
    x="colonne",    # Colonne pour l'axe X
    y="colonne",    # Colonne pour l'axe Y
    color="col",    # Colonne qui définit les couleurs
    title="...",    # Titre du graphique
    labels={...}    # Noms personnalisés des axes
)
```

Les types disponibles : `bar`, `line`, `scatter`, `pie`, `sunburst`, `box`, `histogram`, `heatmap`, `choropleth`…

In [ ]:
# Comparaison des types de graphiques disponibles dans Plotly Express
import plotly.express as px

types_graphiques = {
    "px.bar()": "Barres verticales ou horizontales — comparer des catégories",
    "px.line()": "Lignes — évolution dans le temps",
    "px.scatter()": "Nuage de points — relation entre 2 variables numériques",
    "px.pie()": "Camembert — répartition en parties d'un tout",
    "px.sunburst()": "Soleil hiérarchique — répartition multi-niveaux",
    "px.box()": "Boîte à moustaches — distribution et quartiles",
    "px.histogram()": "Histogramme — distribution d'une variable continue",
    "px.heatmap()": "Carte de chaleur — matrice de valeurs numériques",
    "px.area()": "Aires empilées — évolution de plusieurs catégories cumulées",
    "px.funnel()": "Entonnoir — taux de conversion étape par étape",
}

print("Types de graphiques Plotly Express :")
print()
for func, description in types_graphiques.items():
    print(f"  {func:<20} → {description}")

### Le chaînage de méthodes (method chaining)

Un pattern fréquent dans le code pandas est le **chaînage** : appeler plusieurs méthodes les unes à la suite avec des points. C'est équivalent à créer des variables intermédiaires, mais plus lisible.

In [ ]:
# Comparaison : sans chaînage vs avec chaînage
df = pd.read_csv(DATA + "production_industrielle.csv")

# --- Sans chaînage : variables intermédiaires ---
df_filtre = df[df["ligne"] == "Ligne_1"]
df_groupe = df_filtre.groupby("operateur")["quantite_defauts"].mean()
df_trie = df_groupe.sort_values(ascending=False)
resultat_sans_chaine = df_trie.reset_index()

# --- Avec chaînage : même résultat, code plus compact ---
resultat_avec_chaine = (
    df
    [df["ligne"] == "Ligne_1"]                    # Filtre
    .groupby("operateur")["quantite_defauts"]      # Groupement
    .mean()                                        # Agrégation
    .sort_values(ascending=False)                  # Tri
    .reset_index()                                 # Remise à plat de l'index
)

print("Résultat identique avec ou sans chaînage :")
print(resultat_avec_chaine.to_string(index=False))
print()
print(f"Les deux DataFrames sont égaux : {resultat_sans_chaine.equals(resultat_avec_chaine)}")

---

## Résumé — Quel graphique pour quel type de données ?

Ce tableau de décision aide à choisir le bon type de visualisation selon la nature des données et la question posée.

| Question analytique | Type de données | Graphique recommandé | Fonction Plotly |
|---------------------|-----------------|---------------------|------------------|
| Comparer des catégories (valeurs absolues) | 1 var. catégorielle + 1 var. numérique | **Barres** | `px.bar()` |
| Comparer des catégories (proportions) | 1 var. catégorielle + 1 var. numérique | **Camembert** | `px.pie()` |
| Évolution dans le temps | Dates + 1 var. numérique | **Lignes** | `px.line()` |
| Relation entre 2 variables numériques | 2 var. numériques | **Nuage de points** | `px.scatter()` |
| Distribution d'une variable continue | 1 var. numérique | **Histogramme** | `px.histogram()` |
| Distribution avec quartiles | 1 var. numérique + catégories | **Boîte à moustaches** | `px.box()` |
| Répartition hiérarchique | Plusieurs var. catégorielles + 1 numérique | **Sunburst** | `px.sunburst()` |
| Corrélations multiples | Matrice de variables numériques | **Carte de chaleur** | `px.imshow()` |
| Évolution cumulée de catégories | Dates + catégories + numérique | **Aires empilées** | `px.area()` |
| Comparaison multi-métriques | Plusieurs var. numériques + catégories | **Barres groupées** | `px.bar(barmode="group")` |

---

**Prochaine étape** : Module 1.2 — Aller plus loin avec Claude Code : automatisation, exports, dashboards Streamlit.

---

*Formation Data Mastery — Module 1.1 : Premiers pas avec Claude Code*  
*Niveau : Débutant | Durée estimée : 90 minutes | Prérequis : Notebooks 01 et 02*